# verify02: painful(A) と 遷移BERT(B) と 新モデルD を比べる

全モデルが `dataset/input_pairs.csv` の**同じ採用ペア**を使い、前処理・出力・fold・ハイパラも共通。
違うのは「BERTに入れるテキストの作り方」と「学習データの集め方」だけ。

| モデル | 学習データ | 入力テキスト |
|---|---|---|
| **A. painful** | 痛みノードのみ（162例） | 採用ペア（質問+回答まるごと） |
| **B. 遷移(質問+ペア)** | 3ノード全部（486例・共有1本） | **質問** + 採用ペア（質問+回答まるごと） |
| **D. 遷移(質問+cos回答)** | 3ノード全部（486例・共有1本） | **質問** + そのcos回答（**回答だけ**） |

**Dの狙い**：モデルAと同じ素直な分類器の形のまま、入力を「ノードの質問＋その質問にcosで対応づいた
回答（回答列だけ）」にして、**全データ(486例)＝Aの3倍**で1本の遷移BERTを学習する。
回答部分はラベル均衡を基準に作られていて文章として一貫しないことがあるが、cosが正しく回答を
取れている前提なら、大量データで遷移BERTを学習できる、という設計。

> 採用ペア列：`_1`=痛み / `_2`=しびれ / `_3`=振る舞い。出力はどれも 0=はい/1=いいえ/2=不明 の3クラス。
> B と D の違いは **入力に使う列だけ**（B=「ペア」列＝質問+回答 / D=「回答」列のみ）。

# 1. セットアップ（GPUは git clone / ローカルはそのまま）

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'
IN_COLAB = 'google.colab' in sys.modules


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', 'input_pairs.csv')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.46.3', 'sentencepiece', 'fugashi',
                    'ipadic', 'unidic-lite', 'protobuf', 'accelerate'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', 'input_pairs.csv')
YAML_PATH = os.path.join(REPO_DIR, 'transition_diagram', 'protocol.yaml')
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

## 1.1 実験設定（A/B/D 共通）

In [ ]:
MODEL_NAME = 'cl-tohoku/bert-base-japanese-v3'
MAX_LEN = 256        # CPUで重ければ 64
EPOCHS = 3           # CPUで重ければ 1
LR = 2e-5
BATCH = 8
N_FOLDS = 5
USE_STOPWORDS = False
SEED = 42

import random, numpy as np


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f'MAX_LEN={MAX_LEN} EPOCHS={EPOCHS} LR={LR} BATCH={BATCH} N_FOLDS={N_FOLDS}')

# 2. データ読み込みと前処理（painful 流用 / 入力に使う列を選べるよう拡張）

`make_patient_table` は **どの列をテキストにするか**を `text_col` で選べる：
- `text_col='ペア'` … 質問+回答まるごと（painful / モデルB）
- `text_col='回答'` … cosで対応づいた回答だけ（モデルD）

In [ ]:
import pandas as pd
import yaml

df = pd.read_csv(CSV_PATH)
print('rows:', len(df), '/ patients:', df['id'].nunique())
assert '回答' in df.columns and 'ペア' in df.columns

_proto = yaml.safe_load(open(YAML_PATH, encoding='utf-8'))
_h = next(p for p in _proto['protocols'] if p['id'] == 'headache')
_q = {n['id']: n['question'] for n in _h['nodes']
      if 'choices' in n and not n.get('metadata_only', False)}

NODES = [
    {'key': '痛み',   'adopt': '採用ペア_ひし形_全通り_頭痛_1', 'label': '痛み',   'question': _q['headache_sudden_severe']},
    {'key': 'しびれ', 'adopt': '採用ペア_ひし形_全通り_頭痛_2', 'label': 'しびれ', 'question': _q['headache_numbness_paralysis']},
    {'key': '振る舞い', 'adopt': '採用ペア_ひし形_全通り_頭痛_3', 'label': '振る舞い', 'question': _q['headache_abnormal_behavior']},
]

import fugashi
_tagger = fugashi.Tagger()
STOPWORD_EXTRA_WORDS = set(['の', 'は', 'を', 'に', 'が', 'で', 'と', 'も', 'から', 'より',
                            'へ', 'や', 'など', 'ので', 'けど', 'けれど', '、', '。', 'です', 'ます'])


def remove_stopwords(text):
    return ''.join(w.surface for w in _tagger(text) if w.surface not in STOPWORD_EXTRA_WORDS)


def make_patient_table(adopt_col, label_col, use_question=False, question='', text_col='ペア'):
    """患者ごとに、採用ペア行の text_col を連結して1テキストにする。
    use_question=True なら先頭にノードの質問文を付ける。"""
    rows = []
    for pid, g in df.groupby('id'):
        adopted = g[g[adopt_col] == True]
        parts = adopted[text_col].astype(str).tolist()
        text = ' '.join(parts) if parts else '(発話なし)'
        if USE_STOPWORDS:
            text = remove_stopwords(text)
        if use_question:
            text = question + ' ' + text
        rows.append({'id': pid, 'text': text, 'label': int(g[label_col].iloc[0])})
    return pd.DataFrame(rows).set_index('id')


# 確認：痛みノードで「ペア」と「回答だけ」の違いを見る
_p = make_patient_table(NODES[0]['adopt'], '痛み', use_question=True, question=NODES[0]['question'], text_col='ペア')
_a = make_patient_table(NODES[0]['adopt'], '痛み', use_question=True, question=NODES[0]['question'], text_col='回答')
print('\n[B用] 質問+ペア :', _p['text'].iloc[0][:90])
print('[D用] 質問+回答 :', _a['text'].iloc[0][:90])

# 3. ★BERTの中身★（学習と予測。A/B/D 共通で使う）

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification


class PainTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length,
                             padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


_tok_cache = {}
def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


def build_model(name):
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


# ===== 学習：テキストとラベル → 学習済みBERT =====
def train_model(texts, labels):
    set_seed(SEED)                              # 1) 乱数固定
    tokenizer = get_tokenizer(MODEL_NAME)
    model = build_model(MODEL_NAME)             # 2) まっさらな3クラスBERT
    loader = DataLoader(PainTextDataset(texts, labels, tokenizer, MAX_LEN),
                        batch_size=BATCH, shuffle=True)   # 3) 文章→トークン→バッチ
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    for epoch in range(EPOCHS):                 # 4) 予測→loss→逆伝播→更新 を繰り返す
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
    return model, tokenizer


# ===== 予測：テキスト → 一番スコアの高いクラス =====
@torch.no_grad()
def predict_labels(model, tokenizer, texts):
    model.eval()
    preds = []
    ds = PainTextDataset(texts, [0] * len(texts), tokenizer, MAX_LEN)
    for batch in DataLoader(ds, batch_size=BATCH, shuffle=False):
        batch.pop('labels')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        preds += model(**batch).logits.argmax(dim=-1).cpu().tolist()
    return preds


print('train_model / predict_labels 定義（A/B/D 共通コア）')

# 4. fold分割（患者単位5-fold・A/B/D 共通）

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

patients = np.array(sorted(df['id'].unique()))
triage = np.array([int(df[df['id'] == p]['トリアージ'].iloc[0]) for p in patients])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = [(patients[tr], patients[te]) for tr, te in skf.split(patients, triage)]
print(f'{N_FOLDS}-fold 作成。test患者数:', [len(te) for _, te in FOLDS])

# 5. モデルA：painful（痛みノード単体・採用ペア）

In [ ]:
tabA = make_patient_table(NODES[0]['adopt'], NODES[0]['label'], use_question=False, text_col='ペア')

accA, f1A = [], []
for i, (tr_ids, te_ids) in enumerate(FOLDS):
    model, tok = train_model(tabA.loc[tr_ids, 'text'].tolist(), tabA.loc[tr_ids, 'label'].tolist())
    yp = predict_labels(model, tok, tabA.loc[te_ids, 'text'].tolist())
    yt = tabA.loc[te_ids, 'label'].tolist()
    accA.append(accuracy_score(yt, yp))
    f1A.append(f1_score(yt, yp, average='macro', zero_division=0))
    print(f'  fold{i}: acc={accA[-1]:.3f} f1={f1A[-1]:.3f}')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f'A. painful 痛み: acc={np.mean(accA):.3f}±{np.std(accA):.3f} f1={np.mean(f1A):.3f}±{np.std(f1A):.3f}')

# 6. モデルB / D：遷移BERT（3ノード共有・全486例で1本学習）

`text_col` を切り替えるだけで B（質問+ペア）と D（質問+回答だけ）になる。
学習データは3ノードを全部つなげた**486例**（=Aの3倍）。評価はノードごと。

In [ ]:
def run_transition(text_col, tag):
    # 各ノードの患者別テキスト表（質問は必ず付ける）
    tabs = {n['key']: make_patient_table(n['adopt'], n['label'], use_question=True,
                                         question=n['question'], text_col=text_col)
            for n in NODES}
    per_node = {n['key']: {'acc': [], 'f1': []} for n in NODES}

    for i, (tr_ids, te_ids) in enumerate(FOLDS):
        # 学習データ = 3ノード分を全部つなげる（486例 = 162×3 の train分）
        train_texts, train_labels = [], []
        for n in NODES:
            t = tabs[n['key']]
            train_texts += t.loc[tr_ids, 'text'].tolist()
            train_labels += t.loc[tr_ids, 'label'].tolist()
        model, tok = train_model(train_texts, train_labels)   # ← 共通コアで1本だけ学習

        line = []
        for n in NODES:
            t = tabs[n['key']]
            yt = t.loc[te_ids, 'label'].tolist()
            yp = predict_labels(model, tok, t.loc[te_ids, 'text'].tolist())
            per_node[n['key']]['acc'].append(accuracy_score(yt, yp))
            per_node[n['key']]['f1'].append(f1_score(yt, yp, average='macro', zero_division=0))
            line.append(f"{n['key']}={per_node[n['key']]['acc'][-1]:.3f}")
        print(f'  [{tag}] fold{i}: ' + ' '.join(line))
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return per_node


print('=== B. 遷移BERT（質問+ペア）===')
resB = run_transition(text_col='ペア', tag='B')
print('=== D. 遷移BERT（質問+cos回答だけ）===')
resD = run_transition(text_col='回答', tag='D')
print('done')

# 7. 結果：ノード別 A / B / D 比較（accuracy / macro-F1）

In [ ]:
def ms(v):
    return f'{np.mean(v):.3f} ± {np.std(v):.3f}'


rows_acc, rows_f1 = [], []
for n in NODES:
    k = n['key']
    rows_acc.append({'ノード': k,
                     'A.painful(痛みのみ/ペア)': ms(accA) if k == '痛み' else '—',
                     'B.遷移(質問+ペア)': ms(resB[k]['acc']),
                     'D.遷移(質問+回答だけ)': ms(resD[k]['acc'])})
    rows_f1.append({'ノード': k,
                    'A.painful(痛みのみ/ペア)': ms(f1A) if k == '痛み' else '—',
                    'B.遷移(質問+ペア)': ms(resB[k]['f1']),
                    'D.遷移(質問+回答だけ)': ms(resD[k]['f1'])})

acc_table = pd.DataFrame(rows_acc)
f1_table = pd.DataFrame(rows_f1)
print('===== accuracy ====='); display(acc_table)
print('===== macro-F1 ====='); display(f1_table)

acc_table.to_csv(os.path.join(OUT_DIR, 'verify02_accuracy.csv'), index=False, encoding='utf-8-sig')
f1_table.to_csv(os.path.join(OUT_DIR, 'verify02_f1.csv'), index=False, encoding='utf-8-sig')
print('\nsaved: verify02_accuracy.csv, verify02_f1.csv')
print('参考: painful論文 best run_052(痛み) accuracy=0.716')

# 8. まとめ（読み方）

- **A vs B/D の痛みノード**：単体(A) vs 全データ共有(B/D)。データを増やす（3ノード486例）効果を見る。
- **B vs D**：入力が「質問+ペア(質問+回答)」か「質問+回答だけ」か。
  D は回答列だけを使うので、**質問にcosで対応づいた回答そのもの**で学習する素直な形。
- 3ノード全部を1本で学習しているので、D は「大量データで学習した遷移BERT」に相当する。
- 全部 fold・ハイパラ・採用ペアは共通なので、差は**入力テキストの作り方**だけから来る。